# Memory-Based Agent



A memory-based agent reuses prior context such as preferences, state, or earlier messages.



## Stack

- Framework: LangGraph agent with in-memory checkpointing

- LLM: local Ollama via `ChatOllama(model="llama3.1:latest")`

- Pattern goal: continuity across turns



## When to use it

- Personalization matters

- Tasks span multiple turns or sessions

## Architecture



```mermaid

graph LR

    U[User Message] --> A[Agent]

    A --> M[Memory Checkpointer]

    M --> A

    A --> R[Personalized Reply]

```

In [ ]:
from visual_diagram_helper import render_svg_flow



nodes = [

    ("user", "User Request", 60, 170, "#bfdbfe"),

    ("agent", "Memory Agent", 290, 170, "#bbf7d0"),

    ("store", "Preference Store", 520, 170, "#fde68a"),

    ("prompt", "Personalized Prompt", 750, 170, "#fecaca"),

    ("resp", "Response", 980, 170, "#ddd6fe"),

]

edges = [

    ("user", "agent"),

    ("agent", "store"),

    ("store", "prompt"),

    ("prompt", "resp"),

]



render_svg_flow("Memory Agent Visual Architecture", nodes, edges, width=1220, height=340)

In [2]:
from agent_patterns_common import append_history, get_local_llm, invoke_with_retry, log_event





class PreferenceMemory:

    def __init__(self):

        self.store: dict[str, dict[str, str]] = {}



    def save(self, user_id: str, key: str, value: str) -> None:

        self.store.setdefault(user_id, {})[key] = value

        log_event("info", "memory_saved", user_id=user_id, key=key, value=value)



    def load_all(self, user_id: str) -> dict[str, str]:

        return self.store.get(user_id, {})





memory = PreferenceMemory()

memory.save("arif", "seat_preference", "aisle")

memory.save("arif", "tone", "concise")



llm = get_local_llm()

history = []

preferences = memory.load_all("arif")

history = append_history(history, f"Loaded preferences: {preferences}")

prompt = (

    "Use stored preferences to craft a recommendation.\n"

    "Return EXACTLY 3 bullet points.\n"

    "Each bullet must start with '- '.\n"

    f"Preferences: {preferences}\n"

    f"History: {history}\n"

    "Request: Recommend travel options for Istanbul."

)



response = invoke_with_retry(llm, prompt)

{

    "preferences": preferences,

    "history": history,

    "response": response,

}

{"level": "INFO", "event_type": "memory_saved", "user_id": "arif", "key": "seat_preference", "value": "aisle"}
{"level": "INFO", "event_type": "memory_saved", "user_id": "arif", "key": "tone", "value": "concise"}
{"level": "INFO", "event_type": "llm_invoke_start", "attempt": 1, "prompt_preview": "Use stored preferences to craft a recommendation.\nReturn EXACTLY 3 bullet points"}
{"level": "INFO", "event_type": "llm_invoke_success", "attempt": 1}


{'preferences': {'seat_preference': 'aisle', 'tone': 'concise'},
 'history': ["Loaded preferences: {'seat_preference': 'aisle', 'tone': 'concise'}"],
 'response': 'Here are three travel recommendations based on the stored preferences:\n\n- **Aisle Seat Flights**: Consider booking flights with aisle seats to ensure a comfortable journey, especially during long-haul flights from Istanbul.\n- **Concise Travel Guides**: For a more efficient trip planning experience, opt for concise travel guides that provide essential information without unnecessary details, such as the Lonely Planet\'s "Pocket Guide" series.\n- **Istanbul Airport Lounge Access**: Take advantage of lounge access at Istanbul Airport to enjoy a peaceful atmosphere and complimentary amenities before your flight, perfect for those who value convenience and comfort.'}

## Design insight



Memory works best when it is explicit infrastructure. Keep the storage boundary clear so you can audit what the agent remembers and why.